In [46]:
# [Cell 1] Environment, logging, and reproducibility
# Purpose: Set up device, seeds, output dirs, and a rotating logger with a clear GPU banner.
# Outputs: Console banner with GPU info; ./output/diffusion/pipeline.log created/rotated.

import os, sys, random, json, logging
from pathlib import Path
from logging.handlers import RotatingFileHandler

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import torch_geometric as pyg
    from torch_geometric.utils import to_networkx, dense_to_sparse, from_networkx
    from torch_geometric.data import Data
except Exception as e:
    raise RuntimeError("PyG (torch_geometric) is required. Install a build matching your torch+CUDA.") from e

import networkx as nx

# Output directories
OUT_DIR = Path("./output/diffusion")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Logger
logger = logging.getLogger("track1")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = RotatingFileHandler(OUT_DIR / "pipeline.log", maxBytes=2_000_000, backupCount=3)
    handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(handler)
    console = logging.StreamHandler(sys.stdout)
    console.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(console)

# Device & seeds (deterministic where feasible)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 13
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

gpu_msg = f" | GPU: {torch.cuda.get_device_name(0)} | CUDA: {torch.version.cuda}" if DEVICE == "cuda" else ""
banner = f"🚀 Track 1 — Conditional Graph Diffusion | Device: {DEVICE}{gpu_msg}"
print(banner); logger.info(banner)


🚀 Track 1 — Conditional Graph Diffusion | Device: cuda | GPU: NVIDIA GeForce RTX 4060 Ti | CUDA: 12.6
🚀 Track 1 — Conditional Graph Diffusion | Device: cuda | GPU: NVIDIA GeForce RTX 4060 Ti | CUDA: 12.6


In [47]:
# [Cell 2] Configuration (inline defaults + optional YAML override)
# Purpose: Centralize hyperparameters and analysis toggles. YAML (config.yaml) can override any field.

from typing import Dict, Any

CONFIG: Dict[str, Any] = {
    "device": DEVICE,
    "seed": SEED,
    "epochs": 600,
    "learning_rate": 2e-3,
    "hidden_dim": 128,
    "lambda_motif": 0.04,     # ↓ from 0.20 — prevents over-closing triangles
    "lambda_deg":   0.08,     # keep degree spectrum important
    "lambda_clust": 0.04,     # match mean clustering (but lightly)
    "lambda_assort":0.06,     # NEW: degree assortativity (avoid -0.7 collapse)
    "lambda_apl":   0.06,     # NEW: avg path length (pull back toward ~3.2)
    "lambda_spectral": 0.02,  # keep small but present; uses NumPy eigs on subsample
    "enable_spectral_loss": True,
    "enable_ema": True,         # EMA for stabler sampling
    "ema_decay": 0.999,
    "target_density": None,     # None => infer from real graph
    "gen_num_nodes": 120,       # generated subgraph size
    "gen_steps": 100,           # denoising steps
    "gen_knn": 8,               # kNN backbone during sampling
    "gen_eps": 1e-3,            # tiny re-injected noise
    "report_topk_eigs": 12,     # spectral insight report (subgraphs)
    "analysis_trials": 8,       # number of real subgraph samples
    "loss_sample_n": 400,
    "analysis_sample_n": 120,   # real subgraph sample size
    "paths": {
        "graph_gml": "./merged_master_graph.gml",                # Phase A/B master graph
        "lineage_pkl": "./lineage_embeddings.pkl",               # Phase A lineage/latent
        "motif_csv": "./motif_summary.csv",                      # Phase B motifs per neuron
        "phaseC_csv": "./phaseC_condition_features.csv",         # Phase C condition channels (optional)
    }
}

# Optional YAML override
try:
    import yaml
    cfg_path = Path("./config.yaml")
    if cfg_path.exists():
        with open(cfg_path, "r") as f:
            y = yaml.safe_load(f) or {}
        def deep_update(d, u):
            for k, v in u.items():
                if isinstance(v, dict):
                    d[k] = deep_update(d.get(k, {}), v)
                else:
                    d[k] = v
            return d
        CONFIG = deep_update(CONFIG, y)
        logger.info("Loaded overrides from config.yaml")
except Exception as e:
    logger.warning(f"YAML override skipped: {e}")

print(json.dumps(CONFIG, indent=2))


Loaded overrides from config.yaml
{
  "device": "cuda",
  "seed": 13,
  "epochs": 500,
  "learning_rate": 0.002,
  "hidden_dim": 128,
  "lambda_motif": 0.04,
  "lambda_deg": 0.08,
  "lambda_clust": 0.04,
  "lambda_assort": 0.06,
  "lambda_apl": 0.06,
  "lambda_spectral": 0.02,
  "enable_spectral_loss": true,
  "enable_ema": true,
  "ema_decay": 0.999,
  "target_density": null,
  "gen_num_nodes": 120,
  "gen_steps": 100,
  "gen_knn": 8,
  "gen_eps": 0.001,
  "report_topk_eigs": 12,
  "analysis_trials": 8,
  "loss_sample_n": 400,
  "analysis_sample_n": 120,
  "paths": {
    "graph_gml": "./merged_master_graph.gml",
    "lineage_pkl": "./lineage_embeddings.pkl",
    "motif_csv": "./motif_summary.csv",
    "phaseC_csv": "./phaseC_condition_features.csv"
  },
  "output_dir": "/output/",
  "data_dir": "/data/",
  "batch_size": 32
}


In [48]:
# [Cell 3] Utilities: scaling, NaN‑safe thresholding, kNN builder, stats, and spectral helper
# Purpose: Robust helpers used across training, generation, and analysis.

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

def zscore_df(df: pd.DataFrame) -> pd.DataFrame:
    x = df.astype(float).copy()
    return (x - x.mean(0)) / (x.std(0) + 1e-8)

def triangle_count_undirected(G: nx.Graph) -> float:
    return float(sum(nx.triangles(G).values()) / 3)

def degree_histogram_counts(G: nx.Graph, bins=(0,1,2,3,4,6,8,12,16,24,48,1e9)) -> torch.Tensor:
    deg = np.array([d for _, d in G.degree()])
    hist, _ = np.histogram(deg, bins=bins)
    return torch.tensor(hist / (hist.sum() + 1e-8), dtype=torch.float)

def mmd_l2(p: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
    return torch.mean((p - q)**2)

def choose_threshold_by_density_nansafe(logits: torch.Tensor, undirected_density: float) -> torch.Tensor:
    """
    Pick a threshold for the upper triangle to hit the desired undirected density.
    NaN/Inf‑safe with sensible fallbacks.
    """
    n = logits.shape[0]
    logits = torch.nan_to_num(logits, nan=-1e9, posinf=1e9, neginf=-1e9)
    iu = torch.triu_indices(n, n, offset=1, device=logits.device)
    vals = logits[iu[0], iu[1]]
    vals = vals[torch.isfinite(vals)]
    if vals.numel() == 0:
        return torch.tensor(float('inf'), device=logits.device)
    k_pairs = max(1, int(undirected_density * (n * (n - 1) / 2)))
    k_pairs = min(k_pairs, vals.numel())
    thr = torch.topk(vals, k_pairs).values.min()
    if not torch.isfinite(thr):
        thr = torch.quantile(vals, 0.995)  # sparse fallback
    return thr

def symmetric_adjacency_from_threshold(logits: torch.Tensor, thr: torch.Tensor) -> torch.Tensor:
    A = (logits >= thr).float()
    A.fill_diagonal_(0)
    A = torch.triu(A, diagonal=1)
    A = A + A.T
    return A

def approx_avg_path_length(G: nx.Graph, samples: int = 64) -> float:
    if G.number_of_nodes() == 0:
        return 0.0
    nodes = list(G.nodes())
    pick = nodes if len(nodes) <= samples else random.sample(nodes, samples)
    dists = []
    for s in pick:
        lengths = nx.single_source_shortest_path_length(G, s)
        if len(lengths) > 1:
            dists.extend(list(lengths.values()))
    return float(np.mean(dists)) if dists else 0.0

def topk_norm_laplacian_eigs_numpy(G: nx.Graph, k: int = 12):
    """
    Spectral insight without SciPy:
    Build dense adjacency for subgraphs (<= ~1000 nodes recommended), then compute
    normalized Laplacian eigenvalues via NumPy (eigvalsh on symmetric matrix).
    """
    n = G.number_of_nodes()
    if n == 0:
        return None
    idx = {u:i for i,u in enumerate(G.nodes())}
    A = np.zeros((n, n), dtype=float)
    for u, v in G.edges():
        i, j = idx[u], idx[v]
        A[i, j] = 1.0; A[j, i] = 1.0
    d = A.sum(1)
    with np.errstate(divide='ignore'):
        dinv2 = np.where(d > 0, 1.0/np.sqrt(d), 0.0)
    Dinv2 = np.diag(dinv2)
    L = np.eye(n) - Dinv2 @ A @ Dinv2
    try:
        w = np.linalg.eigvalsh(L)
        return np.sort(w)[:k]
    except Exception:
        return None

def build_knn_edge_index(x: torch.Tensor, k: int) -> torch.Tensor:
    """
    Build a symmetric kNN backbone for message passing (row-wise closest neighbors).
    """
    with torch.no_grad():
        # Use cosine-like distance by normalizing; fallback to Euclidean if norms are zero.
        xn = x / (x.norm(dim=1, keepdim=True) + 1e-8)
        d = torch.cdist(xn, xn)  # [N, N]
        idx = torch.argsort(d, dim=1)[:, 1:k+1]  # skip self
        N = x.size(0)
        rows = torch.arange(N, device=x.device).unsqueeze(1).expand(-1, k)
        ei = torch.stack([rows.reshape(-1), idx.reshape(-1)], dim=0)
        # make undirected
        ei = torch.cat([ei, ei.flip(0)], dim=1)
        return ei


In [49]:
# [Cell 4] Data preparation with attribute-clean graph + Phase A/B/C features
# Purpose: Load graph, strip node attributes, align features by node order, z-score blocks,
#          record condition columns, and save only plain tensors (no pickled Data objects).
# Outputs: PyG Data, z-scored blocks, metadata for conditioning.

from typing import List, Tuple

ALL_COLS: List[str] = []
COND_COLS: List[str] = []
BLOCK_SIZES: dict = {}

def prepare_data() -> Tuple[Data, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    paths = CONFIG["paths"]

    # Load original graph and strip node attributes
    G_raw = nx.read_gml(paths["graph_gml"])
    G = nx.Graph()
    G.add_nodes_from(G_raw.nodes())
    G.add_edges_from(G_raw.edges())

    node_ids = list(G.nodes())

    # Load feature blocks and align to node order
    lineage = pd.read_pickle(paths["lineage_pkl"]).reindex(node_ids)
    motifs  = pd.read_csv(paths["motif_csv"], index_col=0).reindex(node_ids)

    cond = None
    if Path(paths["phaseC_csv"]).exists():
        cond = pd.read_csv(paths["phaseC_csv"], index_col=0).reindex(node_ids).fillna(0.0)

    lineage_z = zscore_df(lineage)
    motifs_z  = zscore_df(motifs)
    if cond is not None and cond.shape[1] > 0:
        cond_z = zscore_df(cond)
    else:
        cond_z = pd.DataFrame(0.0, index=node_ids, columns=["cond_neutral"])

    # Metadata for conditioning
    global ALL_COLS, COND_COLS, BLOCK_SIZES
    lineage_cols = list(lineage_z.columns)
    motif_cols   = list(motifs_z.columns)
    cond_cols    = list(cond_z.columns)
    ALL_COLS  = lineage_cols + motif_cols + cond_cols
    COND_COLS = cond_cols
    BLOCK_SIZES = {"lineage": len(lineage_cols), "motifs": len(motif_cols), "conds": len(cond_cols)}

    # Stack features into PyG Data
    X = np.concatenate([lineage_z.values, motifs_z.values, cond_z.values], axis=1)
    x = torch.tensor(X, dtype=torch.float)
    data = from_networkx(G)
    data.x = x

    # Save only tensors (no pickle of Data object)
    torch.save({"x": data.x.cpu(), "edge_index": data.edge_index.cpu()},
               OUT_DIR / "graph_input.pt")

    msg = {
        "n_nodes": data.num_nodes,
        "n_edges": data.num_edges if hasattr(data, "num_edges") else "unknown",
        "feat_dim": data.x.shape[1],
        "blocks": BLOCK_SIZES,
        "cond_cols": COND_COLS[:10] + (["..."] if len(COND_COLS) > 10 else [])
    }
    logger.info(f"[prepare_data] {msg}")
    print(msg)

    assert torch.isfinite(data.x).all(), "Non-finite values in feature matrix."
    return data, lineage_z, motifs_z, cond_z


In [50]:
# [Cell 5] Model: shallow GAT denoiser + optional EMA wrapper
# Purpose: Predict denoised node features; EMA provides smoother sampling weights.

class GraphDiffusionModel(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, steps: int = 100):
        super().__init__()
        self.in_dim = in_dim
        self.hidden_dim = hidden_dim
        self.out_dim = out_dim
        self.gat1 = pyg.nn.GATConv(in_dim, hidden_dim, heads=4, concat=True, dropout=0.25)
        self.gat2 = pyg.nn.GATConv(hidden_dim*4, out_dim, heads=1, concat=True, dropout=0.25)
        self.register_buffer("noise_schedule", torch.linspace(0.02, 0.15, steps))

    def forward(self, x, edge_index, t: int):
        h = F.elu(self.gat1(x, edge_index))
        y = self.gat2(h, edge_index)
        return y

class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.clone().detach() for k, v in model.state_dict().items()}

    def update(self, model: nn.Module):
        with torch.no_grad():
            for k, v in model.state_dict().items():
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)

    def copy_to(self, model: nn.Module):
        with torch.no_grad():
            model.load_state_dict(self.shadow, strict=True)


In [51]:
# [Cell 6] Training (balanced graph losses: triangles + degree + clustering + assort + APL + spectral)
from tqdm.auto import tqdm

def _subgraph_stats(G: nx.Graph, eig_k: int = 12):
    """Compute compact stats on a graph (used for both real & generated)."""
    n = G.number_of_nodes()
    if n == 0:
        return {
            "n": 0, "tri": 0.0, "tri_per_node": 0.0,
            "deg_hist": torch.zeros(11), "clust_mean": 0.0,
            "assort": 0.0, "apl": 0.0, "eigs": None
        }
    tri = triangle_count_undirected(G)
    tri_per_node = tri / max(n, 1)
    dh = degree_histogram_counts(G)  # torch vector (normalized)
    clust_mean = float(np.mean(list(nx.clustering(G).values()))) if n else 0.0
    assort = nx.degree_assortativity_coefficient(G) if G.number_of_edges() > 0 else 0.0
    apl = approx_avg_path_length(G, samples=min(64, n))
    eigs = topk_norm_laplacian_eigs_numpy(G, k=eig_k)
    return {
        "n": n, "tri": tri, "tri_per_node": float(tri_per_node),
        "deg_hist": dh, "clust_mean": float(clust_mean),
        "assort": float(assort), "apl": float(apl), "eigs": eigs
    }

def train_diffusion_model(graph: Data, cfg=CONFIG):
    torch.manual_seed(cfg["seed"])
    device = cfg["device"]

    model = GraphDiffusionModel(graph.x.shape[1], cfg["hidden_dim"], graph.x.shape[1], steps=cfg["gen_steps"]).to(device)
    ema = EMA(model, decay=cfg["ema_decay"]) if cfg["enable_ema"] else None

    graph = graph.to(device)
    model.noise_schedule = model.noise_schedule.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["learning_rate"])

    # Real full graph + node list (for reproducible subsampling)
    G_real_full = to_networkx(graph, to_undirected=True)
    nodes_full = list(G_real_full.nodes())
    N = len(nodes_full)
    sample_n = min(cfg["loss_sample_n"], N)

    # Target density from real graph
    if cfg["target_density"] is None:
        target_density = (2 * G_real_full.number_of_edges()) / (N * (N - 1))
    else:
        target_density = float(cfg["target_density"])

    # Precompute a fixed real eigs reference on one random subsample (stabilizes target)
    real_eigs_ref = None
    if cfg["enable_spectral_loss"]:
        idx0 = np.random.choice(nodes_full, size=sample_n, replace=False)
        real_eigs_ref = topk_norm_laplacian_eigs_numpy(G_real_full.subgraph(idx0), k=cfg["report_topk_eigs"])

    history = {"epoch": [], "node": [], "triN": [], "degMMD": [], "clust": [], "assort": [], "apl": [], "spec": []}

    pbar = tqdm(range(cfg["epochs"]), desc="Training (Track 1, balanced)")
    for epoch in pbar:
        model.train()

        # ---- Denoise step on node features ----
        t_idx = torch.randint(0, len(model.noise_schedule), (1,), device=device).item()
        noise = torch.randn_like(graph.x) * model.noise_schedule[t_idx]
        x_noisy = graph.x + noise
        pred = model(x_noisy, graph.edge_index, t_idx)
        node_loss = torch.mean((pred - graph.x)**2)

        # ---- Build predicted graph via density‑matched thresholding (no grad) ----
        with torch.no_grad():
            logits = pred @ pred.t()
            thr = choose_threshold_by_density_nansafe(logits, undirected_density=target_density)
            A = symmetric_adjacency_from_threshold(logits, thr)
            ei = dense_to_sparse(A)[0]
            G_pred_full = to_networkx(Data(edge_index=ei), to_undirected=True)

        # ---- Matched subsample (same nodes for real & generated this epoch) ----
        sel = np.random.choice(nodes_full, size=sample_n, replace=False)
        G_r = G_real_full.subgraph(sel).copy()
        G_p = G_pred_full.subgraph(sel).copy()

        # ---- Stats for both ----
        R = _subgraph_stats(G_r, eig_k=cfg["report_topk_eigs"])
        P = _subgraph_stats(G_p, eig_k=cfg["report_topk_eigs"])

        # ---- Graph-level losses (balanced) ----
        # triangles per node: smooth L1 on normalized scale
        tri_loss = F.smooth_l1_loss(torch.tensor(P["tri_per_node"], device=device),
                                    torch.tensor(R["tri_per_node"], device=device))

        # degree histogram MMD (L2)
        deg_loss = mmd_l2(P["deg_hist"].to(device), torch.tensor(R["deg_hist"], device=device))

        # clustering mean
        clust_loss = torch.tensor((P["clust_mean"] - R["clust_mean"])**2, dtype=torch.float, device=device)

        # assortativity
        assort_loss = torch.tensor((P["assort"] - R["assort"])**2, dtype=torch.float, device=device)

        # average path length
        apl_loss = torch.tensor((P["apl"] - R["apl"])**2, dtype=torch.float, device=device)

        # spectral (optional, compare to fixed real eig target to reduce jitter)
        spec_loss = torch.tensor(0.0, device=device)
        if cfg["enable_spectral_loss"]:
            if P["eigs"] is not None and real_eigs_ref is not None:
                pe = torch.tensor(P["eigs"], dtype=torch.float, device=device)
                re = torch.tensor(real_eigs_ref, dtype=torch.float, device=device)
                min_k = min(len(pe), len(re))
                spec_loss = torch.mean((pe[:min_k] - re[:min_k])**2)

        # ---- Weighted total ----
        loss = (node_loss
                + cfg["lambda_motif"] * tri_loss
                + cfg["lambda_deg"]   * deg_loss
                + cfg["lambda_clust"] * clust_loss
                + cfg["lambda_assort"]* assort_loss
                + cfg["lambda_apl"]   * apl_loss
                + cfg["lambda_spectral"] * spec_loss)

        opt.zero_grad(); loss.backward(); opt.step()
        if ema: ema.update(model)

        # ---- Telemetry ----
        if epoch % 10 == 0:
            pbar.set_postfix(
                node=f"{node_loss.item():.3f}",
                triN=f"{tri_loss.item():.4f}",
                degMMD=f"{deg_loss.item():.3f}",
                clust=f"{clust_loss.item():.4f}",
                assort=f"{assort_loss.item():.4f}",
                apl=f"{apl_loss.item():.4f}",
                spec=f"{spec_loss.item():.4f}",
            )

        history["epoch"].append(epoch)
        history["node"].append(float(node_loss.item()))
        history["triN"].append(float(tri_loss.item()))
        history["degMMD"].append(float(deg_loss.item()))
        history["clust"].append(float(clust_loss.item()))
        history["assort"].append(float(assort_loss.item()))
        history["apl"].append(float(apl_loss.item()))
        history["spec"].append(float(spec_loss.item()))

    # ---- Save model(s) & history ----
    ckpt_path = OUT_DIR / "diffusion_model.pt"
    torch.save(model.state_dict(), ckpt_path)
    if ema:
        torch.save(ema.shadow, OUT_DIR / "diffusion_model_ema.pt")

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(OUT_DIR / "training_history.csv", index=False)

    logger.info(f"[train] saved model → {ckpt_path.name} | inferred_density={target_density:.4f}")
    return model, ema, target_density, hist_df


In [52]:
# [Cell 7] Conditioning utilities
# Purpose: Build masks and clamp condition channels for Basal/NPR10/etc during sampling.

def build_condition_mask(cond_cols: list, all_cols: list):
    return [all_cols.index(c) for c in cond_cols if c in all_cols]

def clamp_conditions(x: torch.Tensor, cond_values: dict, all_cols: list):
    if not cond_values: return x
    for name, val in cond_values.items():
        if name in all_cols:
            j = all_cols.index(name)
            x[:, j] = float(val)
    return x


In [53]:
# [Cell 8] Generation with stabilization (kNN backbone, tiny noise) + telemetry
# Purpose: Produce a synthetic subgraph under a given condition prompt.
#          Uses mean/std of training features loaded from safe tensor dict.

@torch.no_grad()
def generate_subgraph(model: GraphDiffusionModel,
                      ema: EMA,
                      num_nodes: int,
                      target_density: float,
                      cond_prompt: dict = None,
                      cfg=CONFIG):
    device = cfg["device"]

    # Use EMA weights if available
    model_eval = GraphDiffusionModel(model.in_dim, model.hidden_dim, model.out_dim, steps=cfg["gen_steps"]).to(device)
    model_eval.load_state_dict(model.state_dict(), strict=True)
    if ema:
        ema.copy_to(model_eval)
    model_eval.eval()

    # Load saved tensors (safe, no pickled Data object)
    saved = torch.load(OUT_DIR / "graph_input.pt", map_location=device, weights_only=False)
    mu = saved["x"].mean(dim=0, keepdim=True)
    sig = saved["x"].std(dim=0, keepdim=True) + 1e-6

    # Initialize latent x
    x = mu + sig * torch.randn(num_nodes, saved["x"].shape[1], device=device)

    # Apply initial condition clamp
    x = clamp_conditions(x, cond_prompt, ALL_COLS)

    # Initial kNN backbone
    edge_index = build_knn_edge_index(x, k=cfg["gen_knn"])

    steps = len(model_eval.noise_schedule)
    tel = []  # telemetry

    for t in reversed(range(steps)):
        x = model_eval(x, edge_index, t) + (torch.randn_like(x) * cfg["gen_eps"])
        x = clamp_conditions(x, cond_prompt, ALL_COLS)

        logits = x @ x.t()
        thr = choose_threshold_by_density_nansafe(logits, undirected_density=target_density)
        A = symmetric_adjacency_from_threshold(logits, thr)
        edge_index = dense_to_sparse(A)[0]

        if t % 10 == 0 or t == steps - 1 or t == 0:
            m = int(edge_index.shape[1] // 2)
            iso = int((A.sum(1) == 0).sum().item())
            tel.append({"step": int(t), "edges": m, "thr": float(thr.item()), "isolated": iso})

    # Fallback if still empty
    if edge_index.numel() == 0:
        logits = x @ x.t()
        logits = torch.nan_to_num(logits, nan=-1e9, posinf=1e9, neginf=-1e9)
        thr = torch.quantile(logits.flatten(), 0.995)
        A = symmetric_adjacency_from_threshold(logits, thr)
        edge_index = dense_to_sparse(A)[0]
        tel.append({"step": -1, "edges": int(edge_index.shape[1] // 2), "thr": float(thr.item()), "isolated": int((A.sum(1) == 0).sum().item())})

    G = to_networkx(Data(x=x, edge_index=edge_index), to_undirected=True)
    nx.write_gml(G, OUT_DIR / f"generated_subgraph_{('prompted' if cond_prompt else 'neutral')}.gml")
    pd.DataFrame(tel).to_csv(OUT_DIR / f"sampler_telemetry_{('prompted' if cond_prompt else 'neutral')}.csv", index=False)

    logger.info(f"[generate] edges={G.number_of_edges()} | iso={sum(1 for _,d in G.degree() if d==0)} | prompt={list(cond_prompt.keys()) if cond_prompt else None}")
    return G


In [54]:
# [Cell 9] Comparative analysis: generated vs matched real subgraphs (fair, same size)
# Purpose: Put synthetic graphs in context with graph-level statistics and spectral insight (NumPy).

from typing import Dict

def graph_stats(G: nx.Graph, eig_k: int = 12) -> Dict[str, float]:
    tri = triangle_count_undirected(G)
    deg = np.array([d for _, d in G.degree()])
    cc_vals = list(nx.clustering(G).values())
    apl = approx_avg_path_length(G, samples=64)
    assort = nx.degree_assortativity_coefficient(G) if G.number_of_edges() > 0 else 0.0
    eigs = topk_norm_laplacian_eigs_numpy(G, k=eig_k)
    s_gap = float(eigs[1] - eigs[0]) if (eigs is not None and len(eigs) > 1) else np.nan
    return {
        "n": G.number_of_nodes(),
        "m": G.number_of_edges(),
        "density": (2*G.number_of_edges())/(G.number_of_nodes()*(G.number_of_nodes()-1)+1e-8) if G.number_of_nodes()>1 else 0.0,
        "triangles": tri,
        "deg_mean": float(np.mean(deg)) if len(deg) else 0.0,
        "deg_std": float(np.std(deg))  if len(deg) else 0.0,
        "clust_mean": float(np.mean(cc_vals)) if cc_vals else 0.0,
        "apl_approx": apl,
        "assort": float(assort),
        "spec_gap": s_gap
    }

def analyze_fair(real_graph: nx.Graph,
                 gen_graph: nx.Graph,
                 sample_n: int,
                 trials: int,
                 eig_k: int):
    nodes_all = list(real_graph.nodes())
    sample_n = min(sample_n, len(nodes_all))
    reals = []
    for _ in range(trials):
        subset = np.random.choice(nodes_all, size=sample_n, replace=False)
        R = real_graph.subgraph(subset).copy()
        reals.append(R)

    gen_s = graph_stats(gen_graph, eig_k=eig_k)
    real_stats = pd.DataFrame([graph_stats(R, eig_k=eig_k) for R in reals])
    comp = real_stats.mean().to_dict()
    comp = {f"real_{k}": v for k, v in comp.items()}
    comp.update({f"gen_{k}": v for k, v in gen_s.items()})

    df = pd.DataFrame([comp])
    out = OUT_DIR / "motif_comparison.csv"
    df.to_csv(out, index=False)
    logger.info(f"[analyze] {df.to_dict(orient='records')[0]}")
    print(df.T)
    return df, real_stats


In [55]:
# [Cell 10] Insight report generator (Markdown)
# Purpose: Convert stats into narrative deltas and cross-condition comparisons for quick scientific reading.

def render_delta(label, gen_val, real_val, fmt=".3f"):
    delta = gen_val - real_val
    s = f"{label}: gen={gen_val:{fmt}} | real≈{real_val:{fmt}} | Δ={delta:{fmt}}"
    return s

def write_insight_report(tag_to_rows):
    """
    tag_to_rows: dict like
      {
        "Basal":  (df_basal, real_stats_basal),
        "NPR10":  (df_npr,  real_stats_npr),
        ...
      }
    """
    lines = []
    lines.append("# Track 1 — Conditional Graph Diffusion: Insight Report\n")
    lines.append("Generated subgraphs are compared to size‑matched real subgraphs.\n")

    for tag, (df, rs) in tag_to_rows.items():
        row = df.iloc[0]
        lines.append(f"## Condition: {tag}\n")
        pairs = [
            ("density", row["gen_density"], row["real_density"]),
            ("triangles", row["gen_triangles"], row["real_triangles"]),
            ("deg_mean", row["gen_deg_mean"], row["real_deg_mean"]),
            ("deg_std",  row["gen_deg_std"],  row["real_deg_std"]),
            ("clust_mean", row["gen_clust_mean"], row["real_clust_mean"]),
            ("apl_approx", row["gen_apl_approx"], row["real_apl_approx"]),
            ("assort", row["gen_assort"], row["real_assort"]),
            ("spec_gap", row["gen_spec_gap"], row["real_spec_gap"]),
        ]
        for label, gv, rv in pairs:
            lines.append("- " + render_delta(label, gv, rv))

        flags = []
        if row["gen_triangles"] > row["real_triangles"] * 1.1: flags.append("↑ more triangles (triadic closure up)")
        if row["gen_clust_mean"] > row["real_clust_mean"] * 1.1: flags.append("↑ higher clustering (local motif density)")
        if row["gen_deg_std"] > row["real_deg_std"] * 1.1: flags.append("↑ heavier degree tail (hubby)")
        if not flags: flags.append("≈ matches real on key motifs and spectra")
        lines.append("\n**Summary signals:** " + "; ".join(flags) + "\n")

    # Cross‑condition diffs if we have ≥2 conditions
    tags = list(tag_to_rows.keys())
    if len(tags) >= 2:
        lines.append("## Cross‑condition contrasts\n")
        base = tag_to_rows[tags[0]][0].iloc[0]
        for other in tags[1:]:
            row = tag_to_rows[other][0].iloc[0]
            lines.append(f"### {other} − {tags[0]}")
            lines.append("- Δ triangles: " + f"{(row['gen_triangles'] - base['gen_triangles']):.3f}")
            lines.append("- Δ clustering mean: " + f"{(row['gen_clust_mean'] - base['gen_clust_mean']):.3f}")
            lines.append("- Δ degree std (tailiness): " + f"{(row['gen_deg_std'] - base['gen_deg_std']):.3f}")
            lines.append("- Δ spectral gap: " + f"{(row['gen_spec_gap'] - base['gen_spec_gap']):.3f}")
            lines.append("")

    report_path = OUT_DIR / "insight_report.md"
    with open(report_path, "w") as f:
        f.write("\n".join(lines))
    print(f"📝 Insight report saved → {report_path}")


In [56]:
# [Cell 11] End-to-end driver with two prompts
# Purpose: Run full pipeline, generate graphs under Basal/NPR10 prompts,
#          analyze vs. real subgraphs, and write a narrative report.
# Added: warning if only 'cond_neutral' is present (Phase C not loaded).

# 1) Prepare data
graph, lineage_z, motifs_z, cond_z = prepare_data()

# ⚠️ Warn if conditioning is inactive
if COND_COLS == ["cond_neutral"]:
    logger.warning("⚠️ Phase C features not detected. Running in UNCONDITIONAL mode. "
                   "Basal vs NPR10 prompts will have no effect.")
    print("⚠️ WARNING: Phase C file not loaded → only 'cond_neutral' present. "
          "Generation is UNCONDITIONAL. Update CONFIG['paths']['phaseC_csv'] "
          "with the correct file to enable Basal/NPR10 conditioning.")

# 2) Train
model, ema, target_density, hist_df = train_diffusion_model(graph, CONFIG)

# 3) Define prompts (keys must match your Phase C CSV)
BASAL_PROMPT = {"Basal_frac": 1.0, "NPR10_frac": 0.0}
NPR_PROMPT   = {"Basal_frac": 0.0, "NPR10_frac": 1.0}

# 4) Generate graphs
G_basal = generate_subgraph(model, ema,
                            CONFIG["gen_num_nodes"],
                            target_density,
                            cond_prompt=BASAL_PROMPT,
                            cfg=CONFIG)
nx.write_gml(G_basal, OUT_DIR / "generated_subgraph_basal.gml")

G_npr = generate_subgraph(model, ema,
                          CONFIG["gen_num_nodes"],
                          target_density,
                          cond_prompt=NPR_PROMPT,
                          cfg=CONFIG)
nx.write_gml(G_npr, OUT_DIR / "generated_subgraph_npr10.gml")

# 5) Analyze vs real graph samples
G_real = nx.read_gml(CONFIG["paths"]["graph_gml"]).to_undirected()
basal_df, basal_real_stats = analyze_fair(
    real_graph=G_real, gen_graph=G_basal,
    sample_n=CONFIG["analysis_sample_n"],
    trials=CONFIG["analysis_trials"],
    eig_k=CONFIG["report_topk_eigs"]
)
npr_df, npr_real_stats = analyze_fair(
    real_graph=G_real, gen_graph=G_npr,
    sample_n=CONFIG["analysis_sample_n"],
    trials=CONFIG["analysis_trials"],
    eig_k=CONFIG["report_topk_eigs"]
)

# 6) Generate report
write_insight_report({"Basal": (basal_df, basal_real_stats),
                      "NPR10": (npr_df, npr_real_stats)})

print("✅ Track 1 pipeline complete.")


[prepare_data] {'n_nodes': 1633, 'n_edges': 20595, 'feat_dim': 702, 'blocks': {'lineage': 278, 'motifs': 418, 'conds': 6}, 'cond_cols': ['Basal_frac', 'NPR10_frac', 'AttractorK', 'Class_Sensory', 'Class_Inter', 'Class_Motor']}
{'n_nodes': 1633, 'n_edges': 20595, 'feat_dim': 702, 'blocks': {'lineage': 278, 'motifs': 418, 'conds': 6}, 'cond_cols': ['Basal_frac', 'NPR10_frac', 'AttractorK', 'Class_Sensory', 'Class_Inter', 'Class_Motor']}


Training (Track 1, balanced):   0%|          | 0/500 [00:00<?, ?it/s]

[train] saved model → diffusion_model.pt | inferred_density=0.0077
[generate] edges=55 | iso=86 | prompt=['Basal_frac', 'NPR10_frac']
[generate] edges=55 | iso=99 | prompt=['Basal_frac', 'NPR10_frac']
[analyze] {'real_n': 120.0, 'real_m': 57.375, 'real_density': 0.008035714285708657, 'real_triangles': 3.375, 'real_deg_mean': 0.95625, 'real_deg_std': 1.353945581802654, 'real_clust_mean': 0.019439484126984125, 'real_apl_approx': 2.7767694942917425, 'real_assort': 0.044382683174312806, 'real_spec_gap': 2.266349933881052e-16, 'gen_n': 120, 'gen_m': 55, 'gen_density': 0.007703081232487603, 'gen_triangles': 35.0, 'gen_deg_mean': 0.9166666666666666, 'gen_deg_std': 3.4024092770989336, 'gen_clust_mean': 0.08880681818181818, 'gen_apl_approx': 1.8758169934640523, 'gen_assort': -0.6897512155974128, 'gen_spec_gap': 0.3706461053487195}
                            0
real_n           1.200000e+02
real_m           5.737500e+01
real_density     8.035714e-03
real_triangles   3.375000e+00
real_deg_mean   